# 3. Reaction pair and product generation

Input: the passed 4π/2π lists in `Data/ZINC_Data`, optimized reactant structures, and the descriptor map configured below. Output: per-diene product CSVs and a self-reaction CSV. The descriptor map named in the historical source differs from part 2's saved `Descriptors_.pkl`; do not substitute it without a key/feature comparison.

Saved outputs were cleared in this split. The archived source notebook retains the historical outputs and execution counts. Set the job paths before running any calculation.


In [ ]:
from pathlib import Path
import os
import sys

# Resolve the repository root from the kernel working directory, or set CYCLOADD_REPO_ROOT.
repo_root_env = os.environ.get("CYCLOADD_REPO_ROOT")
if repo_root_env:
    REPO_ROOT = Path(repo_root_env).expanduser().resolve()
    if not (REPO_ROOT / "Code").is_dir() or not (REPO_ROOT / "Data").is_dir():
        raise RuntimeError(f"CYCLOADD_REPO_ROOT is not a repository root: {REPO_ROOT}")
else:
    REPO_ROOT = next(
        (parent for parent in (Path.cwd(), *Path.cwd().parents)
         if (parent / "Code").is_dir() and (parent / "Data").is_dir()),
        None,
    )
if REPO_ROOT is None:
    raise RuntimeError("Start Jupyter in this repository or set CYCLOADD_REPO_ROOT")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

import os
import pickle
import numpy as np
import pandas as pd
from rdkit import Chem
from tqdm import tqdm
from Code import cycle_process, logfile_process, xtb_process

SMILES_DIR = os.environ.get("CYCLOADD_SMILES_DIR", r"E:\work\Secondary_Selection\Diene_Ene_Smiles")
PRED_CSV_DIR = os.environ.get("CYCLOADD_PRED_CSV_DIR", r"E:\work\Secondary_Selection\All_possible_DA_ZINC")
DESCRIPTOR_MAP = os.path.join(SMILES_DIR, "all_property_detailSterimol_witharea.pkl")


# 3. Generate the All Cross-Combination Dataset with passed 4π and 2π

In [ ]:
smiles_dir=SMILES_DIR
smiles_csv_dir = SMILES_DIR + "/" + "all_smiles.csv"
target_dir = PRED_CSV_DIR
# target_dir = PRED_BO_CSV_DIR -> Just for Reported BO

In [ ]:
result_diene_csv = pd.read_csv(r"Data/ZINC_Data/target_diene_passed.csv")
result_ene_csv = pd.read_csv(r"Data/ZINC_Data/target_ene_passed.csv")
result_diene_csv = result_diene_csv.loc[~np.isnan(result_diene_csv['E/Hatree'].to_numpy())]
result_ene_csv = result_ene_csv.loc[~np.isnan(result_ene_csv['E/Hatree'].to_numpy())]
dienes = set(result_diene_csv["Index"].tolist())
enes = set(result_ene_csv["Index"].tolist())
len(dienes), len(enes)


In [ ]:
# Run It Just When You Generate All Cross-Combination of Reported Bioorthogonal Reactions
# dienes = [5944, 18472, 11875, 18500, 12881]
# enes = [18429, 18627, 18460, 18619, 18427, 18408, 11950, 18454, 18405,
#         11936, 18502, 18611, 18425, 18583, 11955, 18612, 18420, 18457,
#         18445, 18620, 18402, 18400, 18451, 11948, 18421, 18100, 18613,
#         11932, 11933, 18450, 18558, 18423, 18424, 18447, 18453, 18403,
#         18452, 18426]
# The Result are require from 

In [ ]:
smiles_dict = pd.read_csv(smiles_csv_dir)

all_smiles = smiles_dict["Smiles"].to_list()
all_smiles_ids = smiles_dict['Index'].to_list()
if not os.path.isdir(target_dir):
    os.mkdir(target_dir)
QM_MAP = DESCRIPTOR_MAP
with open(QM_MAP, "rb") as f:
    target_map = pickle.load(f)
def diene_with_multienes(diene_id, file_name=None):
    result_dict = {"Diene":{}, "Diene_Index":{}, "Ene":{}, "Ene_Index":{}, "Product":{}, "Unreverse":{}, "Title":{}, "Structure_Id":{}}
    result_dict_id = 0
    diene = all_smiles[all_smiles_ids.index(diene_id)]
    all_mols = []
    all_mols_name = []
    all_titles = []
    result_dir = target_dir + "/"
    if not os.path.isdir(result_dir):
        os.mkdir(result_dir)
    for ene_id in enes:
        ene = all_smiles[all_smiles_ids.index(ene_id)]
        diene_conf_id = smiles_dict["Stable_conf_id"][diene_id]
        ene_conf_id = smiles_dict["Stable_conf_id"][ene_id]

        diene_mol_dir = smiles_dir + '/mol/smilesid_%.5d.mol' % diene_id
        diene_mol = Chem.MolFromMolFile(diene_mol_dir, removeHs=False)
        diene_log_dir = smiles_dir + "/mol_dft_eng/smilesid_%.5d_%.4d.log" % (diene_id, diene_conf_id)
        diene_log = logfile_process.Logfile(diene_log_dir)
        atom_list = diene_log.symbol_list
        position = diene_log.first_atom_position
        diene_mol = xtb_process.xtb_to_mol(diene_mol, [atom_list], [position], 1)
        # Chem.AllChem.EmbedMultipleConfs(diene_mol, 50)
        assert len(diene_mol.GetConformers()) !=0

        ene_mol_dir = smiles_dir + '/mol/smilesid_%.5d.mol' % ene_id
        ene_mol = Chem.MolFromMolFile(ene_mol_dir, removeHs=False)
        ene_log_dir = smiles_dir + "/mol_dft_eng/smilesid_%.5d_%.4d.log" % (ene_id, ene_conf_id)
        ene_log = logfile_process.Logfile(ene_log_dir)
        atom_list = ene_log.symbol_list
        position = ene_log.first_atom_position
        ene_mol = xtb_process.xtb_to_mol(ene_mol, [atom_list], [position], 1)
        # Chem.AllChem.EmbedMultipleConfs(ene_mol, 50)
        assert len(ene_mol.GetConformers()) !=0
        molgroup = []
        # try:
        molgroup = cycle_process.react(dieno_mol = ene_mol, diene_mol = diene_mol,select_diene=0, FFopt=False)
    
        for structure_id, [eng, mol, title] in enumerate(molgroup):
            diene_atoma, diene_atomb, ene_atomc, ene_atomd, _, title_1 = [int(each) for each in title[:6]]
            diene_key = "%.5d %d %d %d" % (diene_id, 1, diene_atoma, diene_atomb)
            ene_key = "%.5d %d %d %d" % (ene_id, 0, ene_atomc, ene_atomd)
            if diene_key not in target_map.keys() or ene_key not in target_map.keys():
                continue
            Chem.SanitizeMol(mol)
            all_mols.append(mol)
            all_titles.append(title)
            product = Chem.MolToSmiles(mol)
            result_dict["Diene"][result_dict_id] = diene
            result_dict["Diene_Index"][result_dict_id] = diene_id
            result_dict["Ene"][result_dict_id] = ene
            result_dict["Ene_Index"][result_dict_id] = ene_id
            result_dict["Product"][result_dict_id] = product
            result_dict["Diene"][result_dict_id] = diene
            result_dict["Title"][result_dict_id] = " ".join([str(num) for num in title])
            result_dict["Structure_Id"][result_dict_id] = structure_id
            result_dict["Unreverse"][result_dict_id] = title[-1]
            result_dict_id += 1
        # except:
        #     pass
    if file_name == None:
        result_dict_name = result_dir + "Diene_Index_%.5d.csv" % diene_id
    else:
        result_dict_name = result_dir + "%s.csv" % file_name
    result_dict = pd.DataFrame(result_dict)
    result_dict.to_csv(result_dict_name, index=False)





In [ ]:
for diene in tqdm(dienes):
    diene_with_multienes(diene)

In [ ]:
def diene_with_self(diene_indexs, file_name=None):
    result_dict = {"Diene":{}, "Diene_Index":{}, "Ene":{}, "Ene_Index":{}, "Product":{}, "Unreverse":{}, "Title":{}, "Structure_Id":{}}
    result_dict_id = 0
    smiles_csv = pd.read_csv(SMILES_DIR + '/all_smiles.csv')
    smiles_id_dict = {row["Index"]: row["Smiles"] for row_id, row in smiles_csv.iterrows()}
    all_mols = []
    all_titles = []
    target_dir = os.environ.get("CYCLOADD_SELF_REACTION_DIR", r"G:\work\Secondary_Selection\All_possible_DA")
    result_dir = target_dir + "/"
    if not os.path.isdir(result_dir):
        os.mkdir(result_dir)
    for diene_id in diene_indexs:
        # try:
        smiles_dir=SMILES_DIR

        ene_id = diene_id
        diene = smiles_id_dict[diene_id]
        ene = diene
        diene_conf_id = smiles_csv["Stable_conf_id"][diene_id]
        ene_conf_id = smiles_csv["Stable_conf_id"][ene_id]
        if len(cycle_process.find_all_diene(Chem.MolFromSmiles(diene))) == 0:
            continue
        diene_mol_dir = smiles_dir + '/mol/smilesid_%.5d.mol' % diene_id
        diene_mol = Chem.MolFromMolFile(diene_mol_dir, removeHs=False)
        diene_log_dir = smiles_dir + "/mol_dft_eng/smilesid_%.5d_%.4d.log" % (diene_id, diene_conf_id)
        diene_log = logfile_process.Logfile(diene_log_dir)
        atom_list = diene_log.symbol_list
        position = diene_log.first_atom_position
        diene_mol = xtb_process.xtb_to_mol(diene_mol, [atom_list], [position], 1)
        # Chem.AllChem.EmbedMultipleConfs(diene_mol, 50)
        assert len(diene_mol.GetConformers()) !=0

        ene_mol_dir = smiles_dir + '/mol/smilesid_%.5d.mol' % ene_id
        ene_mol = Chem.MolFromMolFile(ene_mol_dir, removeHs=False)
        ene_log_dir = smiles_dir + "/mol_dft_eng/smilesid_%.5d_%.4d.log" % (ene_id, ene_conf_id)
        ene_log = logfile_process.Logfile(ene_log_dir)
        atom_list = ene_log.symbol_list
        position = ene_log.first_atom_position
        ene_mol = xtb_process.xtb_to_mol(ene_mol, [atom_list], [position], 1)
        # Chem.AllChem.EmbedMultipleConfs(ene_mol, 50)
        assert len(ene_mol.GetConformers()) !=0
        molgroup = []
        # try:
        molgroup = cycle_process.react(dieno_mol = ene_mol, diene_mol = diene_mol,select_diene=0, FFopt=False)
    
        for structure_id, [eng, mol, title] in enumerate(molgroup):
            Chem.SanitizeMol(mol)
            all_mols.append(mol)
            all_titles.append(title)
            product = Chem.MolToSmiles(mol)
            result_dict["Diene"][result_dict_id] = diene
            result_dict["Diene_Index"][result_dict_id] = diene_id
            result_dict["Ene"][result_dict_id] = ene
            result_dict["Ene_Index"][result_dict_id] = ene_id
            result_dict["Product"][result_dict_id] = product
            result_dict["Diene"][result_dict_id] = diene
            result_dict["Title"][result_dict_id] = " ".join([str(num) for num in title])
            result_dict["Structure_Id"][result_dict_id] = structure_id
            result_dict["Unreverse"][result_dict_id] = title[-1]
            result_dict_id += 1
            # except:
            #     pass
        # except:
        #     continue
    if file_name == None:
        result_dict_name = result_dir + "Diene_Index_%.5d.csv" % diene_id
    else:
        result_dict_name = result_dir + "%s.csv" % file_name
    result_dict = pd.DataFrame(result_dict)
    result_dict.to_csv(result_dict_name)

In [ ]:
diene_with_self(list(dienes | enes), file_name="diene_self")